In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

In [2]:
health = pd.read_csv("gender_health.csv")
greenspace = pd.read_csv("green_blue.csv")
imd = pd.read_csv("area_wealth.csv")



In [3]:
health.head()

In [4]:
greenspace.head()

In [5]:
imd.head()

In [6]:
# Create new health df and remove unnecessary columns
health_2 = health.drop(columns=['Count', 'Population', 'Lower 95% Confidence Interval', 'Upper 95% Confidence Interval', 'Notes'])

#rename existing columns
health_2 = health_2.rename(columns={
    'Area Code': 'area_code',
    'Local Authority': 'la_name',
    'Health Status': 'health_status',
    'Sex': 'sex',
    'Age-standardised Percentage': 'age_std_%'
})

health_2

In [7]:
# Remove welsh LA's as other data only has England LA's
health_2 = health_2[health_2['area_code'].str.startswith('E')].copy()
health_2

In [8]:
# calculate weighted health scores for each sex
weights = {
    'Very good': 5,
    'Good': 4,
    'Fair': 3,
    'Bad': 2,
    'Very bad': 1
}
#Map the weights and add a new column with the weighted calculations
health_2['weights'] = health_2['health_status'].map(weights)
health_2['weighted_score'] = (health_2['age_std_%'] * health_2['weights']) / 100

health_2

In [9]:
# sum up the weighted scores to have one combined score per sex for each LA
health_scores = health_2.groupby(['area_code', 'la_name', 'sex'])['weighted_score'].sum().reset_index()

# create pivot to combine area code/names into 1 row each, with columns for each status and sex combo weighted scores
health_scores = health_scores.pivot_table(
    index = ['area_code', 'la_name'],
    columns = 'sex',
    values = 'weighted_score'
).reset_index()

# rename the weighted score columns
health_scores.columns = ['area_code', 'la_name', 'female_hs', 'male_hs', 'all_hs']

health_scores

In [10]:
# create pivot to combine area code/names into 1 row each, with columns for each status and sex combo percentage value
health_perc = health_2.pivot_table(
    index = ['area_code', 'la_name'],
    columns = ['health_status', 'sex'],
    values = 'age_std_%'
).reset_index()

# flatten the pivot into columns and rename them
health_perc.columns = [
    f'{health_s}_{sex}'.lower()
    .replace('very good', 'v_gd')
    .replace('very bad', 'v_bd')
    .replace(' ', '_')
    .replace('persons', 'all')
    if health_s else sex
    for health_s, sex in health_perc.columns
]

health_perc

In [11]:
health_perc = health_perc.rename(columns={
    'area_code_': 'area_code',
    'la_name_': 'la_name'
})
health_perc

In [12]:
health_full = health_scores.merge(health_perc, on = ['area_code', 'la_name'])
health_full

In [20]:
health_full.describe()

,female_hs,male_hs,all_hs,bad_female,bad_male,bad_all,fair_female,fair_male,fair_all,good_female,good_male,good_all,v_bd_female,v_bd_male,v_bd_all,v_gd_female,v_gd_male,v_gd_all
count,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00,307.00
mean,4.22,4.24,4.23,4.21,3.78,4.01,13.04,12.70,12.88,34.10,34.15,34.12,1.18,1.15,1.17,47.46,48.21,47.83
std,0.08,0.08,0.08,1.08,0.97,1.02,1.75,1.60,1.67,1.49,1.42,1.45,0.37,0.36,0.36,3.77,3.45,3.60
min,4.00,4.05,4.04,2.30,1.90,2.20,8.90,8.90,9.00,26.90,27.00,26.90,0.60,0.60,0.60,39.60,40.90,40.20
25%,4.16,4.18,4.17,3.40,3.00,3.20,11.80,11.50,11.60,33.25,33.40,33.30,0.90,0.90,0.90,44.55,45.45,45.05
50%,4.23,4.25,4.24,4.00,3.70,3.90,13.00,12.50,12.80,34.30,34.30,34.30,1.10,1.10,1.10,47.40,48.30,47.80
75%,4.29,4.30,4.29,4.90,4.40,4.65,14.30,14.00,14.20,35.10,35.05,35.05,1.40,1.40,1.40,50.05,50.65,50.40
max,4.42,4.43,4.42,7.80,6.60,7.00,16.90,16.50,16.60,37.60,37.70,37.50,2.80,2.30,2.50,57.80,58.10,58.00


****Green & Blue Space Data****

In [14]:
# create new green blue space df without the unnecersarry columns
greenspace_2 = greenspace[['LA_Code20', 'LA_Name20', 'MSOA_Ha', 'MSOA_Km2', 'All_Ages', 'GB_Sp_Ha']].copy()

#rename columns to lowercase and to area name/codes column names match the health dataframe
greenspace_2 = greenspace_2.rename(columns={
    'LA_Code20': 'area_code', 
    'LA_Name20': 'la_name',  
    'MSOA_Ha': 'msoa_ha', 
    'MSOA_Km2': 'msoa_km2', 
    'All_Ages': 'population', 
    'GB_Sp_Ha': 'gb_sp_ha'
})

# convert greenspace hectares into km2
greenspace_2['msoa_gs_km2'] = greenspace_2['gb_sp_ha'] / 100

# sum up all the values of the same LAs
greenspace_la = greenspace_2.groupby(['area_code', 'la_name']).agg(
    total_km2 = ('msoa_km2', 'sum'),
    total_pop = ('population', 'sum'),
    total_gs_km2 = ('msoa_gs_km2', 'sum')
).reset_index()

# calculate the greenspace percentage for each LA and add as a new column
greenspace_la['gs_%'] = (greenspace_la['total_gs_km2'] / greenspace_la['total_km2']) * 100

# calculate population density in case it will be useful for a robustness analysis later
greenspace_la['pop_density'] = (greenspace_la['total_pop'] / greenspace_la['total_km2'])

greenspace_la

**IMD Data Frame**

In [15]:
imd

In [16]:
# create new cut down imd df
imd_2 = imd[['Local Authority District code (2019)', 'Local Authority District name (2019)', 'Index of Multiple Deprivation (IMD) Score']].copy()

#rename columns
imd_2 = imd_2.rename(columns = {
    'Local Authority District code (2019)': 'area_code', 
    'Local Authority District name (2019)': 'la_name', 
    'Index of Multiple Deprivation (IMD) Score': 'imd_score'
})

#combine LA's by getting the mean average of imd scores
imd_la = imd_2.groupby(['area_code', 'la_name']).agg(
    imd_score = ('imd_score', 'mean')
).reset_index()


imd_la

**Auto-bots Combine!!**

In [17]:
# combine all 3 dataframes into a big shiny robot collage
# first the health-bot and the green-bot
optimus_maximus = health_full.merge(greenspace_la, on = 'area_code', suffixes = ('', '_gs'))

# now add in Optimus money-bot
optimus_maximus = optimus_maximus.merge(imd_la, on = 'area_code', suffixes = ('', '_imd'))

optimus_maximus

In [18]:
optimus_maxiless = optimus_maximus.drop(columns = ['la_name_imd', 'la_name_gs'])
optimus_maxiless

In [19]:
# Reorder the df to improve readability 
allspark = [
    'area_code', 'la_name',
    # green stats
    'total_km2', 'total_gs_km2', 'gs_%', 
    # population stats
    'total_pop', 'pop_density', 'imd_score',
    #weighted health scores
    'female_hs', 'male_hs', 'all_hs',
    # Very good %
    'v_gd_female', 'v_gd_male', 'v_gd_all',
    # Good %
    'good_female', 'good_male', 'good_all',
    # Fair %
    'fair_female', 'fair_male', 'fair_all',
    # Bad %
    'bad_female', 'bad_male', 'bad_all',
    # Very bad %
    'v_bd_female', 'v_bd_male', 'v_bd_all',
]

optimus_optimal = optimus_maxiless[allspark]

optimus_optimal

,area_code,la_name,total_km2,total_gs_km2,gs_%,total_pop,pop_density,imd_score,female_hs,male_hs,all_hs,v_gd_female,v_gd_male,v_gd_all,good_female,good_male,good_all,fair_female,fair_male,fair_all,bad_female,bad_male,bad_all,v_bd_female,v_bd_male,v_bd_all
0,E06000001,Hartlepool,93.86,5.42,5.77,93663,997.88,34.85,4.08,4.11,4.09,42.30,43.40,42.80,33.70,33.70,33.70,15.90,15.30,15.60,6.20,5.80,6.00,1.90,1.90,1.90
1,E06000002,Middlesbrough,53.88,4.36,8.10,140980,2616.76,40.44,4.09,4.13,4.11,43.00,44.50,43.80,33.40,33.20,33.30,15.10,14.80,14.90,6.50,5.60,6.10,2.00,1.90,2.00
2,E06000003,Redcar and Cleveland,244.90,46.21,18.87,137150,560.03,29.84,4.12,4.14,4.14,43.50,44.70,44.10,34.20,33.70,34.00,14.90,14.70,14.90,5.70,5.20,5.40,1.70,1.70,1.70
3,E06000004,Stockton-on-Tees,203.93,17.72,8.69,197348,967.72,25.24,4.16,4.18,4.17,45.40,46.10,45.70,33.80,33.70,33.70,14.20,14.10,14.20,5.10,4.50,4.90,1.40,1.60,1.50
4,E06000005,Darlington,197.48,5.54,2.81,106803,540.84,26.79,4.18,4.19,4.19,45.10,45.80,45.50,35.00,34.50,34.80,14.00,14.10,14.00,4.70,4.30,4.50,1.20,1.20,1.20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,E09000028,Southwark,28.85,3.40,11.79,318830,11049.89,26.04,4.20,4.25,4.22,48.40,50.50,49.40,31.30,30.90,31.10,13.60,12.90,13.30,5.00,4.30,4.60,1.70,1.40,1.60
298,E09000029,Sutton,43.85,4.22,9.63,206349,4705.75,13.82,4.26,4.28,4.28,48.40,49.40,48.90,35.20,34.70,35.00,11.80,11.70,11.80,3.60,3.10,3.40,1.00,1.00,1.00
299,E09000030,Tower Hamlets,19.77,2.63,13.29,324745,16427.60,27.97,4.00,4.10,4.05,41.10,43.90,42.60,31.40,32.60,32.00,16.90,15.00,15.90,7.80,6.20,7.00,2.80,2.30,2.50
300,E09000031,Waltham Forest,38.82,3.67,9.45,276983,7135.94,25.14,4.16,4.21,4.18,45.00,46.90,45.90,34.40,34.20,34.30,14.10,13.30,13.70,4.90,4.20,4.60,1.60,1.30,1.50
